In [ ]:
 !pip install --upgrade numpy matplotlib scikit-learn seaborn contourpy --quiet
 !pip install --upgrade xgboost timm lion-pytorch --quiet
 !pip install grad-cam shap --quiet

!pip install lime captum
 print(" All packages upgraded! Restart kernel now.")

In [ ]:

import os
import shutil
import pickle
import gc
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.amp import autocast, GradScaler
from torchvision import datasets, transforms
import torchvision.models as tv_models
from torch.utils.data import DataLoader,Subset
from torch.optim.lr_scheduler import CosineAnnealingLR
import timm
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_predict
import xgboost as xgb
from tqdm import tqdm
import warnings


import shap
from pytorch_grad_cam import GradCAM, ScoreCAM  
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget



import torch.nn.functional as F


try:
    from lion_pytorch import Lion
    LION_AVAILABLE = True
    print(" Lion optimizer available")
except ImportError:
    LION_AVAILABLE = False
    print(" lion-pytorch not found")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print("✅ cudnn.benchmark=True, TF32 enabled")

def clear_gpu():
    if device.type == 'cuda':
        torch.cuda.empty_cache()
        gc.collect()

In [ ]:
BATCH_SIZE   = 96
NUM_WORKERS  = 4
PIN_MEMORY   = True
PREFETCH     = 2
merged_dir=r'/rose_testing/rose/Merged_Rose_Dataset'

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])


train_full = datasets.ImageFolder(merged_dir, transform=train_transforms)
val_full   = datasets.ImageFolder(merged_dir, transform=val_transforms)

num_classes  = len(train_full.classes)
class_names  = train_full.classes
dataset_size = len(train_full)


indices       = torch.randperm(dataset_size,
                               generator=torch.Generator().manual_seed(42)).tolist()
train_size    = int(0.8 * dataset_size)
train_indices = indices[:train_size]
val_indices   = indices[train_size:]

train_dataset         = Subset(train_full, train_indices)   
val_dataset           = Subset(val_full,   val_indices)     
train_dataset_ordered = Subset(val_full,   train_indices)   


sample_img, sample_label = train_dataset[0]
print(f"Type: {type(sample_img)}, Shape: {sample_img.shape}, Label: {sample_label}")
assert isinstance(sample_img, torch.Tensor), "FAILED: still returning PIL Image!"
print("Sanity check PASSED")


loader_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                 pin_memory=PIN_MEMORY, prefetch_factor=PREFETCH,
                 persistent_workers=False) 

train_loader = DataLoader(train_dataset, shuffle=True,
                          drop_last=True, **loader_kw)
val_loader   = DataLoader(val_dataset, shuffle=False, **loader_kw)
train_loader_ordered = DataLoader(train_dataset_ordered,
                                  shuffle=False, **loader_kw)

print(f"Classes ({num_classes}): {class_names}")
print(f"Train: {len(train_indices)}  |  Val: {len(val_indices)}")
print(f"Batch: {BATCH_SIZE}  |  Workers: {NUM_WORKERS}")

In [ ]:

EPOCHS       = 10
LION_LR      = 3e-4
LION_WD      = 0.01
GRAD_CLIP    = 1.0
FREEZE_RATIO = 0.7
PATIENCE     = 3

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)


model_registry = {
    
    'CAFormerS18':    'caformer_s18.sail_in22k_ft_in1k',
    'TinyViT':        'tiny_vit_11m_224.dist_in22k_ft_in1k',
    'DaViTTiny':      'davit_tiny.msft_in1k',
    'EfficientNetV2': 'tf_efficientnetv2_s.in21k_ft_in1k',
}

def freeze_early_layers(model, ratio=FREEZE_RATIO):
    params = list(model.parameters())
    for p in params[:int(len(params) * ratio)]:
        p.requires_grad = False

@torch.inference_mode()
def evaluate_model(model, loader):
    model.eval()
    total_loss = correct = total = 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with autocast("cuda"):
            logits = model(x)
            loss = criterion(logits, y)
        total_loss += loss.item() * y.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)
    return total_loss / total, correct / total

@torch.inference_mode()
def extract_features_and_probs(model, loader, desc=""):
    model.eval()
    feats, probs, labels = [], [], []
    for x, y in tqdm(loader, desc=desc, leave=False):
        x = x.to(device, non_blocking=True)
        with autocast("cuda"):
            f = model.forward_features(x)
            logits = model.forward_head(f)
            p = F.softmax(logits.float(), dim=1)
            if   f.dim() == 4: f = F.adaptive_avg_pool2d(f, 1).flatten(1)
            elif f.dim() == 3: f = f.mean(dim=1)
        feats.append(f.cpu().float().numpy())
        probs.append(p.cpu().numpy())
        labels.append(y.numpy())
    return (np.concatenate(feats),
            np.concatenate(probs),
            np.concatenate(labels))


trained_models   = {}
training_history = {}
train_features, val_features = {}, {}
train_probs_cache, val_probs_cache = {}, {}
individual_accuracies = {}
y_train = y_val = None

for name, timm_name in model_registry.items():
    print(f"\n{'='*60}\n  {name}\n{'='*60}")


    model = timm.create_model(timm_name, pretrained=True,
                              num_classes=num_classes).to(device)
    if hasattr(model, 'set_grad_checkpointing'):
        model.set_grad_checkpointing(enable=True)
    freeze_early_layers(model)

    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = (Lion(trainable, lr=LION_LR, weight_decay=LION_WD)
                 if LION_AVAILABLE
                 else optim.AdamW(trainable, lr=1e-3,
                                  weight_decay=LION_WD))
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS,
                                  eta_min=LION_LR * 0.01)
    scaler = GradScaler("cuda")

    history = {'train_loss': [], 'train_acc': [],
               'val_loss': [], 'val_acc': [], 'lr': []}
    best_val_acc = 0.0
    best_state   = None
    wait         = 0


    for epoch in range(EPOCHS):
        model.train()
        running_loss = correct = total = 0

        pbar = tqdm(train_loader,
                    desc=f"  Ep {epoch+1}/{EPOCHS}", leave=False)
        for x, y in pbar:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with autocast("cuda"):
                outputs = model(x)
                loss = criterion(outputs, y)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable, GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()
            correct += (outputs.argmax(1) == y).sum().item()
            total   += y.size(0)

            pbar.set_postfix(
                loss=f"{loss.item():.4f}",
                acc=f"{100*correct/total:.1f}%")

        scheduler.step()

        train_loss = running_loss / len(train_loader)
        train_acc  = 100 * correct / total
        val_loss, val_acc_frac = evaluate_model(model, val_loader)
        val_acc = 100 * val_acc_frac

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['lr'].append(optimizer.param_groups[0]['lr'])

        print(f"    train_loss={train_loss:.4f}  train_acc={train_acc:.1f}%"
              f"  |  val_loss={val_loss:.4f}  val_acc={val_acc:.1f}%")

        if val_acc_frac > best_val_acc:
            best_val_acc = val_acc_frac
            best_state = {k: v.cpu().clone()
                          for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f"    ⏹ Early stop at epoch {epoch+1}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        model.to(device)
    print(f"    Best val acc: {best_val_acc*100:.2f}%")

    training_history[name] = history
    del optimizer, scaler, scheduler, best_state
    clear_gpu()

    if hasattr(model, 'set_grad_checkpointing'):
        model.set_grad_checkpointing(enable=False)

    train_features[name], train_probs_cache[name], y_train = \
        extract_features_and_probs(model, train_loader_ordered, f"  {name} train")
    val_features[name], val_probs_cache[name], y_val = \
        extract_features_and_probs(model, val_loader, f"  {name} val")

    individual_accuracies[name] = accuracy_score(
        y_val, np.argmax(val_probs_cache[name], axis=1))


    model.cpu()
    trained_models[name] = model
    clear_gpu()
    print(f"  ✅ {name} done → CPU (softmax acc: {individual_accuracies[name]*100:.2f}%)")

print("\n✅ All CNNs trained & features extracted!")

In [ ]:

scalers = {}
train_scaled, val_scaled = {}, {}
for name in model_registry:
    scalers[name] = StandardScaler()
    train_scaled[name] = scalers[name].fit_transform(train_features[name])
    val_scaled[name] = scalers[name].transform(val_features[name])


X_train_fused = np.concatenate([train_features[n] for n in model_registry], axis=1)
X_val_fused = np.concatenate([val_features[n] for n in model_registry], axis=1)

fused_scaler     = StandardScaler()
X_train_fused_sc = fused_scaler.fit_transform(X_train_fused)
X_val_fused_sc   = fused_scaler.transform(X_val_fused)


weight_sum  = sum(individual_accuracies.values())
cnn_weights = {n: individual_accuracies[n] / weight_sum for n in model_registry}

cnn_wsv_probs = sum(cnn_weights[n] * val_probs_cache[n] for n in model_registry)
wsv_pred = np.argmax(cnn_wsv_probs, axis=1)
wsv_acc  = accuracy_score(y_val, wsv_pred)

print(f"\n  CNN Weighted Soft Voting Accuracy: {wsv_acc*100:.2f}%")

In [ ]:

xgb_params = dict(
    n_estimators     = 1000,
    max_depth        = 6,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    reg_alpha        = 0.1,
    reg_lambda       = 1.0,
    min_child_weight = 3,
    gamma            = 0.1,
    objective        = 'multi:softprob',
    eval_metric      = 'mlogloss',
    tree_method      = 'hist',
    device           = 'cuda',
    random_state     = 42,
    verbosity        = 0,
)

xgb_clfs, xgb_accs = {}, {}

for name in model_registry:
    clf = xgb.XGBClassifier(**xgb_params)
    clf.fit(train_scaled[name], y_train,
            eval_set=[(val_scaled[name], y_val)], verbose=False)
    pred = clf.predict(val_scaled[name])
    acc  = accuracy_score(y_val, pred)
    xgb_clfs[name] = clf
    xgb_accs[name] = acc
    print(f"  XGB-{name:20s} → {acc*100:.2f}%")

xgb_fused = xgb.XGBClassifier(**xgb_params)
xgb_fused.fit(X_train_fused_sc, y_train,
              eval_set=[(X_val_fused_sc, y_val)], verbose=False)
fused_pred = xgb_fused.predict(X_val_fused_sc)
fused_acc  = accuracy_score(y_val, fused_pred)
xgb_accs['Fused'] = fused_acc
print(f"  XGB-{'Fused':20s} → {fused_acc*100:.2f}%")

clear_gpu()

In [ ]:

xgb_cv_params = {**xgb_params, 'n_estimators': 500}

meta_train_probs, meta_val_probs = [], []

for name in model_registry:
    clf_cv = xgb.XGBClassifier(**xgb_cv_params)
    oof = cross_val_predict(clf_cv, train_scaled[name], y_train,
                            cv=5, method='predict_proba', n_jobs=1)
    meta_train_probs.append(oof)
    meta_val_probs.append(xgb_clfs[name].predict_proba(val_scaled[name]))

meta_X_train = np.hstack(meta_train_probs)
meta_X_val   = np.hstack(meta_val_probs)

meta_lr = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
meta_lr.fit(meta_X_train, y_train)
custom_stack_pred = meta_lr.predict(meta_X_val)
custom_stack_acc  = accuracy_score(y_val, custom_stack_pred)

print(f"  Custom Stacking Accuracy: {custom_stack_acc*100:.2f}%")
clear_gpu()


In [ ]:

xgb_w_total = sum(xgb_accs[n] for n in model_registry)
xgb_weights = {n: xgb_accs[n] / xgb_w_total for n in model_registry}

xgb_wsv_probs = sum(
    xgb_weights[n] * xgb_clfs[n].predict_proba(val_scaled[n])
    for n in model_registry)
xgb_wsv_pred = np.argmax(xgb_wsv_probs, axis=1)
xgb_wsv_acc  = accuracy_score(y_val, xgb_wsv_pred)


xgb_fused_probs   = xgb_fused.predict_proba(X_val_fused_sc)
custom_stack_probs = meta_lr.predict_proba(meta_X_val)

grand_sources = {
    'CNN_WSV':         (cnn_wsv_probs,      wsv_acc),
    'XGB_WSV':         (xgb_wsv_probs,      xgb_wsv_acc),
    'XGB_Fused':       (xgb_fused_probs,    fused_acc),
    'Custom_Stacking': (custom_stack_probs,  custom_stack_acc),
}

grand_w_total = sum(a for _, a in grand_sources.values())
grand_probs   = sum((acc / grand_w_total) * probs
                    for probs, acc in grand_sources.values())
grand_pred = np.argmax(grand_probs, axis=1)
grand_acc  = accuracy_score(y_val, grand_pred)


all_results = {
    'Grand Ensemble':         grand_acc,
    'CNN Weighted Soft Vote': wsv_acc,
    'XGB Weighted Soft Vote': xgb_wsv_acc,
    'XGB Early Fusion':       fused_acc,
    'Custom Stacking':        custom_stack_acc,
}

for n in model_registry:
    all_results[f'CNN-{n}'] = individual_accuracies[n]
    all_results[f'XGB-{n}'] = xgb_accs[n]

sorted_results = sorted(all_results.items(),
                        key=lambda x: x[1], reverse=True)

print(f"\n{'='*60}")
print(f"         COMPLETE RESULTS DASHBOARD")
print(f"{'='*60}")
for rank, (method, acc) in enumerate(sorted_results, 1):
    flag = "First" if rank == 1 else "   "
    print(f" {flag} {rank:<3d} {method:<30s} {acc*100:.2f}%")
print(f"{'='*60}\n")

In [ ]:

print("\nGenerating sample predictions ...")

val_iter = iter(val_loader)
sample_inputs, sample_labels = next(val_iter)
sample_gpu = sample_inputs.to(device, non_blocking=True)
sample_np  = sample_labels.numpy()


cnn_batch = np.zeros((sample_inputs.size(0), num_classes))
sample_feats_list = []

for name in model_registry:
    m = trained_models[name].to(device)
    m.eval()
    with torch.inference_mode(), autocast("cuda"):
        f = m.forward_features(sample_gpu)
        logits = m.forward_head(f)
        probs = F.softmax(logits.float(), dim=1).cpu().numpy()
        if   f.dim() == 4: f = F.adaptive_avg_pool2d(f, 1).flatten(1)
        elif f.dim() == 3: f = f.mean(dim=1)
        sample_feats_list.append(f.cpu().float().numpy())
    cnn_batch += cnn_weights[name] * probs
    m.cpu()
    clear_gpu()

cnn_sample_pred = np.argmax(cnn_batch, axis=1)


sample_fused    = np.concatenate(sample_feats_list, axis=1)
sample_fused_sc = fused_scaler.transform(sample_fused)
xgb_sample_pred = xgb_fused.predict(sample_fused_sc)

def imshow_denorm(inp, title=None, color='black'):
    img = inp.numpy().transpose((1, 2, 0))
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img  = np.clip(std * img + mean, 0, 1)
    plt.imshow(img)
    if title:
        plt.title(title, color=color, fontsize=8)
    plt.axis('off')

n_show = min(16, len(sample_inputs))
fig = plt.figure(figsize=(18, 18))
for i in range(n_show):
    fig.add_subplot(4, 4, i + 1, xticks=[], yticks=[])
    true_c = class_names[sample_np[i]]
    cnn_c  = class_names[cnn_sample_pred[i]]
    xgb_c  = class_names[xgb_sample_pred[i]]
    ok     = (true_c == cnn_c == xgb_c)
    imshow_denorm(
        sample_inputs[i],
        title=f"True: {true_c}\nCNN-WSV: {cnn_c}\nXGB: {xgb_c}",
        color='green' if ok else 'red')
plt.suptitle("CNN Weighted-Soft-Vote  vs  XGBoost-Fused",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n {'True':<25} | {'CNN-WSV':<25} | {'XGBoost':<25}")
print(f" {'─'*25}─┼─{'─'*25}─┼─{'─'*25}")
for i in range(n_show):
    t = class_names[sample_np[i]]
    c = class_names[cnn_sample_pred[i]]
    x = class_names[xgb_sample_pred[i]]
    cm = " " if t == c else "*"
    xm = " " if t == x else "*"
    print(f" {t:<25} | {cm}{c:<24} | {xm}{x:<24}")
print("  (* = wrong)\n")

In [ ]:

print("\n" + "=" * 60)
print(f"  GRAND ENSEMBLE — CLASSIFICATION REPORT ({grand_acc*100:.2f}%)")
print("=" * 60)
print(classification_report(y_val, grand_pred, target_names=class_names, digits=4))

print("=" * 60)
print(f"  CUSTOM STACKING — CLASSIFICATION REPORT ({custom_stack_acc*100:.2f}%)")
print("=" * 60)
print(classification_report(y_val, custom_stack_pred, target_names=class_names, digits=4))

fig, axes = plt.subplots(1, 2, figsize=(24, 10))

for ax, (title, preds, acc) in zip(axes, [
    ("Grand Ensemble",  grand_pred,       grand_acc),
    ("Custom Stacking", custom_stack_pred, custom_stack_acc),
]):
    cm = confusion_matrix(y_val, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, cbar_kws={'label': 'Count'})
    ax.set_xlabel('Predicted', fontsize=12, fontweight='bold')
    ax.set_ylabel('True',      fontsize=12, fontweight='bold')
    ax.set_title(f'{title} ({acc*100:.2f}%)', fontsize=13, fontweight='bold')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for name, hist in training_history.items():
    epochs_ran = range(1, len(hist['train_loss']) + 1)
    axes[0, 0].plot(epochs_ran, hist['train_loss'], label=name, lw=2)
    axes[0, 1].plot(epochs_ran, hist['train_acc'],  label=name, lw=2)
    axes[1, 0].plot(epochs_ran, hist['val_loss'],   label=name, lw=2)
    axes[1, 1].plot(epochs_ran, hist['val_acc'],    label=name, lw=2)

for ax, title, ylabel in [
    (axes[0, 0], 'Training Loss',     'Loss'),
    (axes[0, 1], 'Training Accuracy', 'Accuracy (%)'),
    (axes[1, 0], 'Validation Loss',   'Loss'),
    (axes[1, 1], 'Validation Accuracy', 'Accuracy (%)'),
]:
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:

save_dir = 'saved_ensemble_models_rose_v3'
os.makedirs(save_dir, exist_ok=True)

for name, model in trained_models.items():
    p = os.path.join(save_dir, f"{name}_weights.pth")
    torch.save(model.state_dict(), p)
    print(f"  {name:20s} → {p}")

for name, clf in xgb_clfs.items():
    p = os.path.join(save_dir, f"xgb_{name}.json")
    clf.save_model(p)
    print(f"   XGBoost-{name:14s} → {p}")

xgb_fused.save_model(os.path.join(save_dir, "xgb_fused.json"))
print(f"   XGBoost-Fused         → xgb_fused.json")

with open(os.path.join(save_dir, "meta_lr.pkl"), 'wb') as f:
    pickle.dump(meta_lr, f)
print(f"   Meta-LR               → meta_lr.pkl")

with open(os.path.join(save_dir, "scalers.pkl"), 'wb') as f:
    pickle.dump({'per_cnn': scalers, 'fused': fused_scaler}, f)
print(f"   Scalers               → scalers.pkl")

results_pkg = {
    'individual_cnn_acc': individual_accuracies,
    'cnn_weights':        cnn_weights,
    'xgb_accuracies':     xgb_accs,
    'xgb_weights':        xgb_weights,
    'all_results':        dict(sorted_results),
    'class_names':        class_names,
    'training_history':   training_history,
    'config': {
        'optimizer':       'Lion' if LION_AVAILABLE else 'AdamW',
        'batch_size':      BATCH_SIZE,
        'epochs':          EPOCHS,
        'patience':        PATIENCE,
        'lion_lr':         LION_LR,
        'weight_decay':    LION_WD,
        'grad_clip':       GRAD_CLIP,
        'freeze_ratio':    FREEZE_RATIO,
        'label_smoothing': 0.1,
        'models':          list(model_registry.keys()),
    },
}
with open(os.path.join(save_dir, "results.pkl"), 'wb') as f:
    pickle.dump(results_pkg, f)
print(f"   Results & config      → results.pkl")

print(f"\n Everything saved to: {os.path.abspath(save_dir)}")

if device.type == 'cuda':
    allocated = torch.cuda.memory_allocated() / 1024**2
    reserved  = torch.cuda.memory_reserved()  / 1024**2
    peak      = torch.cuda.max_memory_allocated() / 1024**2
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**2
    print(f"\n GPU Memory:")
    print(f"   Current:  {allocated:.0f} / {reserved:.0f} MB")
    print(f"   Peak:     {peak:.0f} MB")
    print(f"   Free:     {total_vram - reserved:.0f} MB")

In [ ]:

print("Generating Grad-CAM explanations for all models...")

NUM_IMAGES_PER_CLASS = 1
NUM_CLASSES_TO_SHOW = 4


selected_images = []
selected_labels = []
class_seen = set()

for images, labels in val_loader:
    for i in range(len(labels)):
        label = labels[i].item()
        if label not in class_seen:
            selected_images.append(images[i])
            selected_labels.append(label)
            class_seen.add(label)

        if len(class_seen) == NUM_CLASSES_TO_SHOW:
            break
    if len(class_seen) == NUM_CLASSES_TO_SHOW:
        break


for model_name in trained_models:

    print(f"\nGrad-CAM for {model_name}")

    model = trained_models[model_name].to(device)
    model.eval()

    target_layer = None
    for module in reversed(list(model.modules())):
        if isinstance(module, torch.nn.Conv2d):
            target_layer = module
            break

    cam = GradCAM(model=model, target_layers=[target_layer])

    plt.figure(figsize=(12, 10))

    for i in range(NUM_CLASSES_TO_SHOW):

        img = selected_images[i]
        label = selected_labels[i]

        input_tensor = img.unsqueeze(0).to(device)

        img_np = img.numpy().transpose(1,2,0)

        mean = np.array([0.485,0.456,0.406])
        std  = np.array([0.229,0.224,0.225])
        rgb_img = np.clip(std * img_np + mean,0,1)

        targets = [ClassifierOutputTarget(label)]

        grayscale_cam = cam(input_tensor=input_tensor,targets=targets)[0]

        visualization = show_cam_on_image(rgb_img,grayscale_cam,use_rgb=True)

        plt.subplot(NUM_CLASSES_TO_SHOW,2,2*i+1)
        plt.imshow(rgb_img)
        plt.title(f"Original\n{class_names[label]}")
        plt.axis("off")

        plt.subplot(NUM_CLASSES_TO_SHOW,2,2*i+2)
        plt.imshow(visualization)
        plt.title(f"GradCAM")
        plt.axis("off")

    plt.suptitle(f"{model_name} Grad-CAM Explanations",fontsize=16)
    plt.tight_layout()
    plt.show()

    model.cpu()
    clear_gpu()

In [ ]:

print("Generating Score-CAM explanations...")

for model_name in trained_models:

    print(f"\nScore-CAM for {model_name}")

    model = trained_models[model_name].to(device)
    model.eval()


    target_layer = None
    for module in reversed(list(model.modules())):
        if isinstance(module, torch.nn.Conv2d):
            target_layer = module
            break

    cam = ScoreCAM(
        model=model,
        target_layers=[target_layer],
    
    )

    plt.figure(figsize=(12,10))

    for i in range(NUM_CLASSES_TO_SHOW):

        img = selected_images[i]
        label = selected_labels[i]

        input_tensor = img.unsqueeze(0).to(device)

        img_np = img.numpy().transpose(1,2,0)

        mean = np.array([0.485,0.456,0.406])
        std  = np.array([0.229,0.224,0.225])

        rgb_img = np.clip(std * img_np + mean,0,1)

        targets = [ClassifierOutputTarget(label)]

        grayscale_cam = cam(
            input_tensor=input_tensor,
            targets=targets
        )[0]

        visualization = show_cam_on_image(
            rgb_img,
            grayscale_cam,
            use_rgb=True
        )

        plt.subplot(2,2,i+1)
        plt.imshow(visualization)
        plt.title(class_names[label])
        plt.axis("off")

    plt.suptitle(f"{model_name} Score-CAM Explanations",fontsize=16)
    plt.tight_layout()
    plt.show()

    model.cpu()
    clear_gpu()

In [ ]:

from lime import lime_image
from skimage.segmentation import mark_boundaries

print("Running LIME explanations...")

explainer = lime_image.LimeImageExplainer()

for model_name in trained_models:

    print(f"\nLIME for {model_name}")

    model = trained_models[model_name].to(device)
    model.eval()

    def batch_predict(images):

        images = torch.tensor(images).permute(0,3,1,2).float().to(device)

        with torch.no_grad():
            preds = model(images)

        return torch.softmax(preds,dim=1).cpu().numpy()

    plt.figure(figsize=(12,10))

    for i in range(NUM_CLASSES_TO_SHOW):

        img = selected_images[i]

        img_np = img.numpy().transpose(1,2,0)

        explanation = explainer.explain_instance(
            img_np,
            batch_predict,
            top_labels=1,
            hide_color=0,
            num_samples=1000
        )

        temp, mask = explanation.get_image_and_mask(
            explanation.top_labels[0],
            positive_only=True,
            num_features=5,
            hide_rest=False
        )

        plt.subplot(2,2,i+1)
        plt.imshow(mark_boundaries(temp/255.0,mask))
        plt.title(class_names[selected_labels[i]])
        plt.axis("off")

    plt.suptitle(f"{model_name} LIME Explanations",fontsize=16)
    plt.tight_layout()
    plt.show()

    model.cpu()

In [ ]:

from captum.attr import IntegratedGradients
import gc

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

print("Running Integrated Gradients...")


for name in trained_models:
    trained_models[name].cpu()
clear_gpu()

for model_name in trained_models:

    print(f"\nIntegrated Gradients for {model_name}")


    clear_gpu()

    model = trained_models[model_name].to(device)
    model.eval()

    ig = IntegratedGradients(model)

    plt.figure(figsize=(12, 10))

    for i in range(NUM_CLASSES_TO_SHOW):

        img = selected_images[i].unsqueeze(0).to(device)
        label = selected_labels[i]

        attr, delta = ig.attribute(
            img,
            target=label,
            n_steps=50,              
            internal_batch_size=1,    
            return_convergence_delta=True
        )

        attr = attr.squeeze().cpu().detach().numpy()
        attr = np.transpose(attr, (1, 2, 0))
        attr = np.mean(np.abs(attr), axis=2)
        attr = (attr - attr.min()) / (attr.max() - attr.min() + 1e-8)

        img_np = selected_images[i].numpy().transpose(1, 2, 0)
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        rgb_img = np.clip(std * img_np + mean, 0, 1)

        plt.subplot(2, 2, i + 1)
        plt.imshow(rgb_img)
        plt.imshow(attr, cmap="jet", alpha=0.5)
        plt.title(class_names[label])
        plt.axis("off")


        del img, attr, delta
        clear_gpu()

    plt.suptitle(f"{model_name} Integrated Gradients", fontsize=16)
    plt.tight_layout()
    plt.show()


    model.cpu()
    del model, ig
    clear_gpu()

In [ ]:

print("Calculating SHAP explanations...")

explainer = shap.TreeExplainer(xgb_fused)

X_val_subset = X_val_fused_sc[:200]
shap_values = explainer.shap_values(X_val_subset)


feature_names = []
model_feature_ranges = {}

start = 0
for name in model_registry:

    num_feats = train_features[name].shape[1]
    end = start + num_feats

    model_feature_ranges[name] = (start,end)

    feature_names.extend([f"{name}_f{i}" for i in range(num_feats)])

    start = end



plt.figure(figsize=(10,6))

shap.summary_plot(
    shap_values,
    X_val_subset,
    feature_names=feature_names,
    plot_type="bar",
    max_display=20,
    show=False
)

plt.title("Top CNN Features Influencing Final Prediction",fontsize=14)
plt.tight_layout()
plt.show()


plt.figure(figsize=(10,8))

shap.summary_plot(
    shap_values,
    X_val_subset,
    feature_names=feature_names,
    max_display=15
)



model_importance = {}

for model,(s,e) in model_feature_ranges.items():
    model_importance[model] = np.abs(shap_values[:,s:e]).mean()

plt.figure(figsize=(8,5))

plt.bar(model_importance.keys(),model_importance.values())
plt.title("Contribution of Each CNN to Final Ensemble",fontsize=14)
plt.ylabel("Mean |SHAP Value|")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:


import torch.nn.functional as F

def generate_masks(N, s, p1, input_size):

    cell_size = np.ceil(np.array(input_size) / s)
    up_size = (s + 1) * cell_size

    masks = np.random.rand(N, s, s) < p1
    masks = masks.astype('float32')

    masks = torch.from_numpy(masks)

    masks = F.interpolate(
        masks.unsqueeze(1),
        size=tuple(up_size.astype(int)),
        mode='bilinear',
        align_corners=False
    )

    masks = masks[:, :, :input_size[0], :input_size[1]]

    return masks.squeeze(1)



def rise_explanation(model, image, label, N=2000, s=8, p1=0.5, batch_size=32):

    model.eval()

    input_size = image.shape[1:]  
    H, W = input_size


    masks = generate_masks(N, s, p1, input_size)  

    image_dev = image.to(device) 

    saliency = torch.zeros(H, W, device='cpu')  


    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)

     
        mask_batch = masks[start:end].to(device)  


        masked_images = image_dev * mask_batch.unsqueeze(1) 
        with torch.no_grad():
            preds = model(masked_images)
            probs = torch.softmax(preds, dim=1)[:, label]   


        probs_cpu = probs.view(-1, 1, 1).cpu()    
        mask_cpu = mask_batch.cpu()                
        saliency += (probs_cpu * mask_cpu).sum(dim=0)


        del mask_batch, masked_images, preds, probs, probs_cpu, mask_cpu
        torch.cuda.empty_cache()


    saliency = saliency.numpy()
    saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)


    del masks, image_dev
    torch.cuda.empty_cache()

    return saliency


In [ ]:

print("Generating RISE explanations...")


for name in trained_models:
    trained_models[name].cpu()
clear_gpu()

for model_name in trained_models:

    print(f"\nRISE for {model_name}")
    clear_gpu()

    model = trained_models[model_name].to(device)
    model.eval()

    plt.figure(figsize=(12, 10))

    for i in range(NUM_CLASSES_TO_SHOW):

        img = selected_images[i]
        label = selected_labels[i]

        saliency = rise_explanation(model, img, label, N=2000, s=8, p1=0.5, batch_size=32)

        img_np = img.numpy().transpose(1, 2, 0)
        mean = np.array([0.485, 0.456, 0.406])
        std  = np.array([0.229, 0.224, 0.225])
        rgb_img = np.clip(std * img_np + mean, 0, 1)

        plt.subplot(2, 2, i + 1)
        plt.imshow(rgb_img)
        plt.imshow(saliency, cmap="jet", alpha=0.5)
        plt.title(class_names[label])
        plt.axis("off")

        del saliency
        clear_gpu()

    plt.suptitle(f"{model_name} RISE Explanations", fontsize=16)
    plt.tight_layout()
    plt.show()

    model.cpu()
    del model
    clear_gpu()

In [ ]:

print("Generating Aggregated XAI Comparison for ALL models...")


for name in trained_models:
    trained_models[name].cpu()
clear_gpu()

for model_name, model in trained_models.items():

    print(f"\nProcessing model: {model_name}")
    model = model.to(device)
    model.eval()


    target_layer = None
    for module in reversed(list(model.modules())):
        if isinstance(module, torch.nn.Conv2d):
            target_layer = module
            break

    if target_layer is None:
        print(f"  Warning: No Conv2d layer found in {model_name}. Grad-CAM / Score-CAM will be skipped.")

    for i in range(NUM_CLASSES_TO_SHOW):

        img = selected_images[i]
        label = selected_labels[i]
        label_idx = int(label) if torch.is_tensor(label) else label

        input_tensor = img.unsqueeze(0).to(device)

        img_np = img.numpy().transpose(1, 2, 0)
        mean = np.array([0.485, 0.456, 0.406])
        std  = np.array([0.229, 0.224, 0.225])
        rgb_img = np.clip(std * img_np + mean, 0, 1)


        grad_vis = np.zeros_like((rgb_img * 255).astype(np.uint8))
        grad_cam_map = None

        if target_layer is not None:
            gradcam = GradCAM(model=model, target_layers=[target_layer])
            targets = [ClassifierOutputTarget(label_idx)]
            grad_cam_map = gradcam(input_tensor=input_tensor, targets=targets)[0]
            grad_vis = show_cam_on_image(rgb_img, grad_cam_map, use_rgb=True)
            del gradcam
            clear_gpu()


        score_vis = np.zeros_like((rgb_img * 255).astype(np.uint8))
        score_cam_map = None

        if target_layer is not None:
            scorecam = ScoreCAM(model=model, target_layers=[target_layer])
            targets = [ClassifierOutputTarget(label_idx)]
            score_cam_map = scorecam(input_tensor=input_tensor, targets=targets)[0]
            score_vis = show_cam_on_image(rgb_img, score_cam_map, use_rgb=True)
            del scorecam
            clear_gpu()


        ig = IntegratedGradients(model)
        attr, _ = ig.attribute(
            input_tensor,
            target=label_idx,
            n_steps=50,
            internal_batch_size=1,
            return_convergence_delta=True
        )
        attr = attr.squeeze().cpu().detach().numpy()
        attr = np.transpose(attr, (1, 2, 0))
        attr = np.mean(np.abs(attr), axis=2)
        attr = (attr - attr.min()) / (attr.max() - attr.min() + 1e-8)
        del ig
        clear_gpu()


        del input_tensor
        clear_gpu()

        rise_map = rise_explanation(model, img, label_idx, N=2000, s=8, p1=0.5, batch_size=32)
        clear_gpu()


        def batch_predict(images):
            batch = torch.tensor(images).permute(0, 3, 1, 2).float()
            preds_list = []

            for start in range(0, len(batch), 32):
                end = min(start + 32, len(batch))
                mini = batch[start:end].to(device)
                with torch.no_grad():
                    preds = model(mini)
                    preds_list.append(torch.softmax(preds, dim=1).cpu())
                del mini, preds
                torch.cuda.empty_cache()

            return torch.cat(preds_list, dim=0).numpy()

        explainer = lime_image.LimeImageExplainer()
        explanation = explainer.explain_instance(
            img_np,
            batch_predict,
            top_labels=1,
            num_samples=1000
        )

        lime_img, mask = explanation.get_image_and_mask(
            explanation.top_labels[0],
            positive_only=True,
            num_features=5,
            hide_rest=False
        )
        lime_vis = mark_boundaries(lime_img / 255.0, mask)
        del explainer, explanation
        clear_gpu()


        plt.figure(figsize=(12, 8))

        plt.subplot(2, 3, 1)
        plt.imshow(rgb_img)
        plt.title("Original")
        plt.axis("off")

        plt.subplot(2, 3, 2)
        if target_layer is not None:
            plt.imshow(grad_vis)
            plt.title("Grad-CAM")
        else:
            plt.imshow(rgb_img)
            plt.title("Grad-CAM (N/A)")
        plt.axis("off")

        plt.subplot(2, 3, 3)
        if target_layer is not None:
            plt.imshow(score_vis)
            plt.title("Score-CAM")
        else:
            plt.imshow(rgb_img)
            plt.title("Score-CAM (N/A)")
        plt.axis("off")

        plt.subplot(2, 3, 4)
        plt.imshow(lime_vis)
        plt.title("LIME")
        plt.axis("off")

        plt.subplot(2, 3, 5)
        plt.imshow(rgb_img)
        plt.imshow(attr, cmap="jet", alpha=0.5)
        plt.title("Integrated Gradients")
        plt.axis("off")

        plt.subplot(2, 3, 6)
        plt.imshow(rgb_img)
        plt.imshow(rise_map, cmap="jet", alpha=0.5)
        plt.title("RISE")
        plt.axis("off")

        plt.suptitle(f"{model_name} XAI Comparison — {class_names[label_idx]}", fontsize=16)
        plt.tight_layout()
        plt.show()


        del grad_vis, score_vis, attr, rise_map, lime_vis
        if grad_cam_map is not None:
            del grad_cam_map
        if score_cam_map is not None:
            del score_cam_map
        clear_gpu()

        print(f"  Completed {class_names[label_idx]} for {model_name}")

    model.cpu()
    clear_gpu()

print("\nAll model comparisons completed.")